# Tutorial: SI-ConvNeXt for directional solidification (AM)

A lightweight demo on how to build, train, and roll out the autoregressive deep surrogate (ADS) of

> Ji et al., *Scalable Autoregressive Deep Surrogates
> for Dendritic Microstructure Dynamics*, arXiv:2511.03884 (2025),
> https://arxiv.org/abs/2511.03884

- ## Section 1: File structure
- ## Section 2: Build the AM SI-ConvNeXt model
- ## Section 3: Training (next-step prediction, $\phi, c$ loss only)
- ## Section 4: Autoregressive rollout with imposed $T$
- ## Section 5: Visualization and validation

Differences from the isothermal tutorial: three channels $(\phi, c, T)$ with $T$ *imposed* via $T = T_0 + G(x - V_{\mathrm{iso}} t)$ (frozen temperature approximation, FTA), **zero-flux** padding, depth $M = 14$.

No data is included; cells that need data print `OK`/`MISS` and skip if files are absent.

# Section 1: File structure

- **NPS package**: `pip install -e .` from <https://github.com/llnl/NPS>, or set `NPS_PATH` in Section 2.
- **Dataset (`data/directional_training.npy`)**: user-supplied, shape `(N_seq, N_t, Ny, Nx, C) = (24, 250, 400, 100, 3)`, channels $(\phi, c, T)$; growth axis `Ny` (size 400, physical $x$), $T$ = imposed FTA field.
- **Checkpoints (`checkpoints/`)**: trained weights (written by Section 3, read by Section 4).
- **Outputs**: `rollout_directional.npy`, `mse_directional.svg`.

# Section 2: Build the AM SI-ConvNeXt model

Three in/out channels $(\phi, c, T)$; the predicted $T$ is overwritten by the imposed field at rollout (Section 4). Depth $M = 14$, hidden dim 64, kernel 3, **zero-flux** padding (Neumann BCs in the transverse direction).

In [ ]:
from pathlib import Path
import sys
import numpy as np
import torch

NPS_PATH = None
if NPS_PATH is not None:
    sys.path.insert(0, str(Path(NPS_PATH).resolve()))

from NPS.model.convnext import ConvNeXtIsotropic

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch =', torch.__version__, ' device =', DEVICE)


In [ ]:
model = ConvNeXtIsotropic(
    in_chans=3, num_classes=3,
    depth=14, dim=64, kernel_size=3,
    DIM=2, periodic=False, zero_flux=True,
).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'AM SI-ConvNeXt params: {n_params:,}')


# Section 3: Training (next-step prediction, $\phi, c$ loss only)

The imposed $T$ is excluded from the loss: 1000 epochs, AdamW. Minimal driver; the manuscript additionally uses Gaussian input noise (no point-group augmentation here, as the thermal gradient breaks rotational invariance).

In [ ]:
def next_step_loss_am(model, frame_n, frame_np1, loss_channels=(0, 1)):
    """MSE on (phi, c); imposed T (channel 2) excluded. Tensors (B, 3, Ny, Nx)."""
    pred = model(frame_n)
    pred_loss = pred[:, list(loss_channels)]
    tgt_loss  = frame_np1[:, list(loss_channels)]
    return torch.mean((pred_loss - tgt_loss) ** 2)

def train_one_step_am(model, optim, frame_n, frame_np1):
    """Single AdamW gradient step (phi, c loss only)."""
    model.train()
    loss = next_step_loss_am(model, frame_n, frame_np1)
    optim.zero_grad(); loss.backward(); optim.step()
    return float(loss)

In [ ]:
import os

DATA_PATH = 'data/directional_training.npy'          # (N_seq, N_t, 400, 100, 3), see Section 1
CKPT_PATH = 'checkpoints/directional_si_convnext.pt'

for f in [DATA_PATH]:
    print(('OK  ' if os.path.exists(f) else 'MISS'), f)

if os.path.exists(DATA_PATH):
    train = np.load(DATA_PATH, mmap_mode='r')        # (N_seq, N_t, Ny, Nx, 3)
    N_seq, N_t = train.shape[:2]
    pairs = [(s, t) for s in range(N_seq) for t in range(N_t - 1)]
    optim = torch.optim.AdamW(model.parameters(), lr=6e-4)   # manuscript sweeps 2e-4 to 8e-4, keeps validation-best
    batch = 16                                       # adjust to your memory budget

    for epoch in range(1000):
        np.random.shuffle(pairs)
        running = 0.0
        for i in range(0, len(pairs), batch):
            s = np.array([p[0] for p in pairs[i:i + batch]])
            t = np.array([p[1] for p in pairs[i:i + batch]])
            frame_n   = torch.from_numpy(np.asarray(train[s, t],     dtype=np.float32)).permute(0, 3, 1, 2).to(DEVICE)
            frame_np1 = torch.from_numpy(np.asarray(train[s, t + 1], dtype=np.float32)).permute(0, 3, 1, 2).to(DEVICE)
            running += train_one_step_am(model, optim, frame_n, frame_np1) * len(s)
        if epoch % 50 == 0:
            print(f'epoch {epoch:4d}  mean next-step MSE {running / len(pairs):.3e}')

    os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)
    torch.save(model.state_dict(), CKPT_PATH)
    print('saved', CKPT_PATH)
else:
    print('Dataset missing -- skipping training. See Section 1 for the expected layout.')

# Section 4: Autoregressive rollout with imposed $T$

Each step keeps the predicted $\phi, c$ and overwrites $T$ with the analytic FTA field at the new time. Set `T0`, `G`, `V_iso` to the values used to generate your data.

In [ ]:
def make_T_field(Ny, Nx, t_step, T0, G, V_iso, growth_axis=0):
    """FTA field T(x, t) = T0 + G (x - V_iso t); growth_axis = array axis of physical x."""
    if growth_axis == 0:
        x = np.arange(Ny, dtype=np.float32)[:, None]
        field = T0 + G * (x - V_iso * t_step)              # (Ny, 1)
        return np.broadcast_to(field, (Ny, Nx)).astype(np.float32)
    else:
        x = np.arange(Nx, dtype=np.float32)[None, :]
        field = T0 + G * (x - V_iso * t_step)              # (1, Nx)
        return np.broadcast_to(field, (Ny, Nx)).astype(np.float32)

@torch.no_grad()
def autoregressive_rollout_with_imposed_T(
        model, ic, n_steps, T0, G, V_iso, growth_axis=0, device=None):
    """Roll out n_steps from ic (Ny, Nx, 3), regenerating T (channel 2) each step.

    Returns (n_steps + 1, Ny, Nx, 3); index 0 is the IC.
    """
    if device is None:
        device = next(model.parameters()).device
    Ny, Nx, _ = ic.shape
    model.eval()
    state = (torch.from_numpy(ic.astype(np.float32))
             .permute(2, 0, 1).unsqueeze(0).to(device))
    out = [state.cpu().squeeze(0).permute(1, 2, 0).numpy()]
    for k in range(1, n_steps + 1):
        state = model(state)
        T_k = torch.from_numpy(
            make_T_field(Ny, Nx, k, T0, G, V_iso, growth_axis=growth_axis)
        ).to(device)
        state[0, 2] = T_k
        out.append(state.cpu().squeeze(0).permute(1, 2, 0).numpy())
    return np.stack(out, axis=0)

In [ ]:
import os

DATA_PATH = 'data/directional_training.npy'
CKPT_PATH = 'checkpoints/directional_si_convnext.pt'

T0, G, V_iso = 0.0, 52.80, 0.12      # FTA parameters: T(x, t) = T0 + G (x - V_iso t)

for f in [DATA_PATH, CKPT_PATH]:
    print(('OK  ' if os.path.exists(f) else 'MISS'), f)

if os.path.exists(DATA_PATH) and os.path.exists(CKPT_PATH):
    model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
    gt = np.load(DATA_PATH, mmap_mode='r')[0]        # first trajectory, (N_t, Ny, Nx, 3)
    rollout = autoregressive_rollout_with_imposed_T(
        model, np.asarray(gt[0]),
        n_steps=len(gt) - 1,
        T0=T0, G=G, V_iso=V_iso,
        growth_axis=0,
    )
    np.save('rollout_directional.npy', rollout)
    print('rollout shape:', rollout.shape, ' -> saved rollout_directional.npy')
else:
    print('Need the dataset and a trained checkpoint (Section 3) to roll out.')

# Section 5: Visualization and validation

### Section 5.1: Ground truth (GT) vs ADS prediction at selected steps

Panels are transposed so the growth axis runs horizontally.

In [ ]:
%matplotlib inline
import os
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = 'data/directional_training.npy'
ROLLOUT_PATH = 'rollout_directional.npy'

for f in [DATA_PATH, ROLLOUT_PATH]:
    print(('OK  ' if os.path.exists(f) else 'MISS'), f)

if os.path.exists(DATA_PATH) and os.path.exists(ROLLOUT_PATH):
    gt  = np.load(DATA_PATH, mmap_mode='r')[0]       # (N_t, Ny, Nx, 3)
    ads = np.load(ROLLOUT_PATH)                      # (n_steps + 1, Ny, Nx, 3)
    timesteps = [0, 50, 150, len(gt) - 1]

    for ch, name in [(0, r'$\phi$'), (1, r'$c$')]:
        fig, axes = plt.subplots(len(timesteps), 2, figsize=(12, 1.6 * len(timesteps)))
        for i, t in enumerate(timesteps):
            axes[i, 0].imshow(gt[t, ..., ch].T, interpolation='nearest')
            axes[i, 0].set_title(f'GT {name}, step {t}', fontsize=10)
            axes[i, 1].imshow(ads[t, ..., ch].T, interpolation='nearest')
            axes[i, 1].set_title(f'ADS {name}, step {t}', fontsize=10)
            for ax in axes[i]:
                ax.axis('off')
        plt.tight_layout()
        plt.show()
else:
    print('Run Sections 3-4 first (or drop in your own rollout).')

### Section 5.2: Validation with per-step normalized MSE

$\mathrm{MSE}^i(t)$ for $i \in \{\phi, c\}$, normalized by the squared initial-field range $r_i^2$ (manuscript Methods); the imposed $T$ is excluded.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = 'data/directional_training.npy'
ROLLOUT_PATH = 'rollout_directional.npy'

for f in [DATA_PATH, ROLLOUT_PATH]:
    print(('OK  ' if os.path.exists(f) else 'MISS'), f)

if os.path.exists(DATA_PATH) and os.path.exists(ROLLOUT_PATH):
    gt  = np.load(DATA_PATH, mmap_mode='r')[0]
    ads = np.load(ROLLOUT_PATH)
    n = min(len(gt), len(ads))
    gt0 = np.asarray(gt[0], dtype=np.float64).reshape(-1, gt.shape[-1])
    r = gt0.max(0) - gt0.min(0)                      # initial-field range r_i, per channel
    mse = ((np.asarray(gt[:n], dtype=np.float64) - ads[:n]) ** 2).mean(axis=(1, 2)) / r**2

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(mse[:, 0], label=r'$\phi$')
    ax.plot(mse[:, 1], label=r'$c$')
    ax.set_xlabel('Rollout step')
    ax.set_ylabel('Normalized MSE')
    ax.set_yscale('log')
    ax.legend(loc='best', frameon=False)
    fig.savefig('mse_directional.svg', format='svg', bbox_inches='tight', pad_inches=0.1)
    plt.show()
else:
    print('Run Sections 3-4 first (or drop in your own rollout).')